# 🌾 KisanAgent — GRPO Training with Unsloth

**Meta-PyTorch OpenEnv Hackathon Finale 2026 — Theme 3: World Modeling**

This notebook trains a Qwen2.5-7B model using GRPO (Group Relative Policy Optimization) on the KisanAgent RL environment.

## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install unsloth trl gymnasium torch matplotlib requests pyyaml openenv-core mergekit datasets
!pip install 'accelerate>=0.26.0'

# Clone the repo
!git clone https://github.com/Gouravbirwaz/meta-hackathon-final-idea /content/meta-hackathon-final-idea

import sys
sys.path.insert(0, '/content/meta-hackathon-final-idea')
print('Dependencies installed.')

## Cell 2 — Initialize Environment

In [ ]:
import sys
REPO_PATH = '/content/meta-hackathon-final-idea'
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)

from server.app import KisanEnvironment
from env.models import KisanAction

env = KisanEnvironment()
print('KisanEnvironment initialized in memory!')

## Cell 3 — Random Agent Baseline

In [ ]:
import random
DECISIONS = ['irrigate', 'fertilize', 'spray_pesticide', 'sell_now', 'hold_crop', 'apply_scheme', 'take_loan', 'do_nothing']

def run_random_episode(difficulty='medium'):
    obs = env.reset(difficulty=difficulty)
    for _ in range(90):
        action = KisanAction(farm_decision=random.choice(DECISIONS), reasoning='random')
        obs = env.step(action=action)
        if obs.done:
            meta = obs.metadata or {}
            return meta.get('net_income_inr', 0), meta.get('final_scores', {})
    return 0, {}

print('Running baseline...')
income, scores = run_random_episode('medium')
print(f'Random Baseline: ₹{income:,.0f} | Score: {scores.get("composite_score", 0):.3f}')

## Cell 4 — Load Model (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=16, lora_dropout=0, bias='none', use_gradient_checkpointing='unsloth', random_state=42)
print('Model loaded with LoRA.')

## Cell 5 — GRPO Reward Function

In [ ]:
import json as _json
SYSTEM_PROMPT = 'You are KisanAgent. Maximize net income across 90 days. Respond ONLY in JSON: {"reasoning": "...", "tool_to_call": "tool_name or null", "farm_decision": "decision or null"}'

def kisan_reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    for completion in completions:
        try:
            text = completion if isinstance(completion, str) else completion[-1].get('content', '')
            parsed = _json.loads(text)
            action = KisanAction(
                farm_decision=parsed.get('farm_decision') if parsed.get('farm_decision') != 'null' else None,
                tool_name=parsed.get('tool_to_call') if parsed.get('tool_to_call') != 'null' else None,
                reasoning=parsed.get('reasoning', '')
            )
            obs = env.step(action=action)
            if obs.done: reward = float((obs.metadata or {}).get('final_scores', {}).get('composite_score', 0.0))
            else: reward = float(obs.reward or 0.0)
        except: reward = 0.0
        rewards.append(reward)
    return rewards
print('Reward function ready.')

## Cell 6 — GRPO Training Loop

In [ ]:
import time, os
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

CURRICULUM = [(0, 150, 'easy'), (150, 350, 'medium'), (350, 500, 'hard')]

def get_model_action(obs, difficulty):
    prompt = f'Day {obs.day}: Stage {obs.crop_stage}, Balance {obs.bank_balance_inr}, Yield {obs.estimated_yield_kg}, Alerts {obs.active_alerts}. Decide:'
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to('cuda')
    with torch.no_grad(): outputs = model.generate(inputs, max_new_tokens=128, temperature=0.1)
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    try:
        p = _json.loads(response)
        return KisanAction(farm_decision=p.get('farm_decision') if p.get('farm_decision') != 'null' else None, tool_name=p.get('tool_to_call') if p.get('tool_to_call') != 'null' else None, reasoning=p.get('reasoning', ''))
    except: return KisanAction(farm_decision='do_nothing')

def run_trained_episode(difficulty, seed=42):
    obs = env.reset(difficulty=difficulty, seed=seed)
    for _ in range(90):
        obs = env.step(action=get_model_action(obs, difficulty))
        if obs.done: return (obs.metadata or {}).get('net_income_inr', 0), (obs.metadata or {}).get('final_scores', {})
    return 0, {}

all_prompts = []
for s, e, d in CURRICULUM:
    for ep in range(s, e):
        obs = env.reset(difficulty=d, seed=ep)
        all_prompts.append({'prompt': [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': f'Day {obs.day}: Balance {obs.bank_balance_inr}'}]})

trainer = GRPOTrainer(model=model, reward_funcs=[kisan_reward_fn], args=GRPOConfig(output_dir='checkpoints', num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=5e-6, num_generations=4, max_new_tokens=256, logging_steps=5, seed=42), train_dataset=Dataset.from_list(all_prompts), processing_class=tokenizer)
print('Starting training...')
trainer.train()

print('\nFinal Evaluation:')
for d in ['easy', 'medium', 'hard']:
    inc, sc = run_trained_episode(d, seed=999)
    print(f'  [{d.upper()}] Income: ₹{inc:,.0f} | Score: {sc.get("composite_score", 0):.3f}')

## Cell 7 — Export to GGUF

In [ ]:
model.save_pretrained_gguf('kisan_model', tokenizer, quantization_method='q4_k_m')